In [2]:
import numpy as np
import pandas as pd
import joblib
import re
from urllib.parse import urlparse

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [3]:
url_model = joblib.load("../model/url_model_calibrated.pkl")
sms_model = joblib.load("../model/sms_model_calibrated.pkl")
vectorizer = joblib.load("../model/tfidf_vectorizer.pkl")

In [4]:
url_df = pd.read_csv("../data/malicious_phish.csv")

sms_df = pd.read_csv("../data/spam.csv", encoding='latin-1')
sms_df = sms_df[['v1','v2']]
sms_df.columns = ['label','message']

In [5]:
sms_df = pd.read_csv("../data/spam.csv", encoding='latin-1')
sms_df = sms_df[['v1', 'v2']]
sms_df.columns = ['label', 'message']
sms_df['label'] = sms_df['label'].map({'ham': 0, 'spam': 1})

In [6]:
def extract_url_features(url):
    parsed = urlparse(url)

    return [
        len(url),
        url.count('.'),
        url.count('-'),
        url.count('@'),
        url.count('?'),
        url.count('='),
        url.count('/'),
        url.count('www'),
        1 if parsed.scheme == 'https' else 0,
        1 if re.match(r"^\d+\.\d+\.\d+\.\d+", parsed.netloc) else 0,
        1 if any(k in url.lower() for k in ["login","verify","bank","secure","account"]) else 0,
        len(parsed.netloc)
    ]

In [7]:
url_features = url_df['url'].apply(lambda x: extract_url_features(x))
url_features = pd.DataFrame(url_features.tolist())

In [8]:
url_df['label'] = url_df['type'].apply(lambda x: 0 if x == 'benign' else 1)
url_labels = url_df['label'].values

In [9]:
sms_features = vectorizer.transform(sms_df['message'])
sms_labels = sms_df['label'].map({'ham':0, 'spam':1}).values

In [10]:
url_probs = url_model.predict_proba(url_features)[:,1]
sms_probs = sms_model.predict_proba(sms_features)[:,1]

In [11]:
n = min(len(url_probs), len(sms_probs))

url_probs = url_probs[:n]
sms_probs = sms_probs[:n]
labels = url_labels[:n]

In [12]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
meta_model.fit(fusion_X, fusion_y)

NameError: name 'fusion_X' is not defined

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:,1]
    
    print("Accuracy:", accuracy_score(y, y_pred))
    print("Precision:", precision_score(y, y_pred))
    print("Recall:", recall_score(y, y_pred))
    print("F1 Score:", f1_score(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, y_prob))
    print("-"*40)

evaluate_model(meta_model, fusion_X, fusion_y)

NameError: name 'fusion_X' is not defined

In [54]:
joblib.dump(meta_model, "../model/fusion_model.pkl")

['../model/fusion_model.pkl']

In [55]:
url_features.shape[1]

12

In [39]:
sms_probs = sms_model.predict_proba(sms_features)[:,1]

In [40]:
len(url_probs) == len(sms_probs)

True

In [41]:
print(len(url_probs), len(sms_probs))

5572 5572


In [14]:
fusion_X = np.vstack((url_probs, sms_probs)).T
fusion_y = y

NameError: name 'y' is not defined

In [38]:
url_probs = url_model.predict_proba(url_features)[:,1]
sms_probs = sms_model.predict_proba(sms_features)[:,1]

ValueError: Feature shape mismatch, expected: 12, got 10

In [1]:
fusion_X = np.vstack((url_probs, sms_probs)).T
fusion_y = labels

NameError: name 'np' is not defined

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
meta_model.fit(fusion_X, fusion_y)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:,1]
    
    print("Accuracy:", accuracy_score(y, y_pred))
    print("Precision:", precision_score(y, y_pred))
    print("Recall:", recall_score(y, y_pred))
    print("F1 Score:", f1_score(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, y_prob))
    print("-"*40)

In [ ]:
print("Fusion Model Results")
evaluate_model(meta_model, fusion_X, fusion_y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    fusion_X, fusion_y, test_size=0.2, random_state=42, stratify=fusion_y
)

In [ ]:
meta_model = LogisticRegression()
meta_model.fit(X_train_f, y_train_f)

In [ ]:
print("Fusion Model (Proper Evaluation)")
evaluate_model(meta_model, X_test_f, y_test_f)

In [ ]:
fusion_weighted = 0.6 * url_probs + 0.4 * sms_probs

In [ ]:
from sklearn.metrics import roc_auc_score

print("ROC-AUC:", roc_auc_score(fusion_y, fusion_weighted))

In [ ]:
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    y_pred = (meta_model.predict_proba(X_test_f)[:,1] >= t).astype(int)
    
    from sklearn.metrics import recall_score, precision_score, f1_score
    
    print(f"Threshold: {t}")
    print("Precision:", precision_score(y_test_f, y_pred))
    print("Recall:", recall_score(y_test_f, y_pred))
    print("F1:", f1_score(y_test_f, y_pred))
    print("-"*30)

In [ ]:
joblib.dump(meta_model, "../model/fusion_model.pkl")

In [ ]:
len(extract_url_features("http://test.com"))